# Практика 2. Работа с LLM через API (GigaChat + LangChain)


## 0. Установка библиотек


In [25]:
import pandas as pd
from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

## 1. Импорт библиотек и подключение к GigaChat

In [26]:
import os
import json
import langchain_gigachat
import langchain_core
import pandas
from dotenv import load_dotenv

from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

load_dotenv()
API_KEY = os.getenv("GIGA_KEY")

if not API_KEY:
    raise ValueError(
        "Не найден GIGA_KEY. Создайте файл .env рядом с ноутбуком "
        "и добавьте строку GIGA_KEY=ваш_ключ."
    )

llm = GigaChat(
    credentials=API_KEY,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

print(llm.invoke("Привет! Ответь коротко.").content)

Привет. Коротко.


## 2. Выбор варианта и загрузка данных



In [27]:
VARIANT = "20"  

csv_path = f"rental_{VARIANT}.csv"

try:
    df = pd.read_csv(csv_path, sep=";", encoding="utf-8-sig")
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, sep=";", encoding="cp1251")

required_columns = {"text", "amount"}
if not required_columns.issubset(df.columns):
    raise ValueError(f"В CSV должны быть столбцы {required_columns}. Найдены: {list(df.columns)}")

if len(df) != 15:
    raise ValueError(f"Ожидается 15 заявок, а найдено {len(df)}.")

df["amount"] = pd.to_numeric(df["amount"], errors="raise").astype(int)
print(f"Загружено заявок: {len(df)}")
display(df.head())

Загружено заявок: 15


,amount,text
0,6,Здравствуйте! Ищем жильё для 2 взрослых и 4 де...
1,2,"Добрый вечер! Ищем жилье на 2 человек, с 3 сен..."
2,4,Ищем жильё на 4 взрослых с 31.08.18 по 07.08.1...
3,3,"Здравствуйте, интересует жилье с 17.08 по 25.0..."
4,3,Добрый день. Ищем комфортное жильё в Лазаревс...


## 3. Задание 1 — базовый промпт

In [28]:
system_prompt = SystemMessagePromptTemplate.from_template(
    "Ты — эксперт по анализу текстовых заявок на аренду жилья. "
    "Определи общее количество людей, которые будут проживать."
)

user_prompt = HumanMessagePromptTemplate.from_template(
    "Заявка: {text}\n\n"
    "Извлеки общее количество проживающих. "
    "Верни только одно целое число, без пояснений."
)

chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])
chain = chat_prompt | llm | StrOutputParser()

print(chain.invoke({"text": "Семья из трех человек снимает квартиру"}))  # ожидается 3

3


## 4. Обработка 15 заявок и расчёт точности

In [29]:
def run_chain(chain_to_run, data, output_column):
    results = []
    for _, row in data.iterrows():
        try:
            answer = chain_to_run.invoke({"text": row["text"]})
            results.append(str(answer).strip())
        except Exception as error:
            results.append(f"ERROR: {error}")
    data[output_column] = results
    return data

def calc_accuracy(data, output_column):
    predicted = pd.to_numeric(data[output_column], errors="coerce")
    correct_mask = predicted == data["amount"]
    correct = int(correct_mask.sum())
    total = len(data)
    accuracy = correct / total if total else 0
    print(
        f"{output_column}: верных {correct} из {total}; "
        f"ошибок {total - correct}; точность {accuracy:.1%}"
    )
    return accuracy

df = run_chain(chain, df, "result")
acc_base = calc_accuracy(df, "result")
display(df[["text", "amount", "result"]])
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")

result: верных 15 из 15; ошибок 0; точность 100.0%


,text,amount,result
0,Здравствуйте! Ищем жильё для 2 взрослых и 4 де...,6,6
1,"Добрый вечер! Ищем жилье на 2 человек, с 3 сен...",2,2
2,Ищем жильё на 4 взрослых с 31.08.18 по 07.08.1...,4,4
3,"Здравствуйте, интересует жилье с 17.08 по 25.0...",3,3
4,Добрый день. Ищем комфортное жильё в Лазаревс...,3,3
5,Ищем жильё в Лазаревское заезд 13 июля на 10 д...,4,4
6,Здравствуйте! Интересует жилье недорого 1-10 с...,4,4
7,Здравствуйте🤗\nИщем жильё в центре рядом с мор...,2,2
8,Здравствуйте! Ищем жильё с 15 по 23 августа 2...,3,3
9,Сниму жилье в июне у моря 2 взрослых до 1500 в...,2,2


## 5. Улучшение промпта — Few-shot


In [30]:
few_shot_system = SystemMessagePromptTemplate.from_template(
    """Ты — эксперт по анализу заявок на аренду жилья.
Определи ОБЩЕЕ количество проживающих: взрослых и детей вместе.

Правила:
- "семья с двумя детьми" = 4 человека (2 взрослых + 2 ребёнка);
- "семья с одним ребёнком" = 3 человека;
- "пара" = 2 человека;
- "один человек" = 1 человек;
- если прямо указано число людей, используй его;
- не путай количество комнат, спальных мест и людей.

Примеры:
Заявка: "Ищем жильё на троих" -> 3
Заявка: "Двое взрослых и четыре ребенка" -> 6
Заявка: "Одна взрослая и ребенок 5 лет" -> 2

Верни ТОЛЬКО целое число, без пояснений."""
)

few_shot_prompt = ChatPromptTemplate.from_messages([few_shot_system, user_prompt])
chain_fs = few_shot_prompt | llm | StrOutputParser()

df = run_chain(chain_fs, df, "result_fewshot")
acc_fs = calc_accuracy(df, "result_fewshot")
print(f"Изменение точности: {acc_base:.1%} -> {acc_fs:.1%}")
display(df[["text", "amount", "result", "result_fewshot"]])

result_fewshot: верных 15 из 15; ошибок 0; точность 100.0%
Изменение точности: 100.0% -> 100.0%


,text,amount,result,result_fewshot
0,Здравствуйте! Ищем жильё для 2 взрослых и 4 де...,6,6,6
1,"Добрый вечер! Ищем жилье на 2 человек, с 3 сен...",2,2,2
2,Ищем жильё на 4 взрослых с 31.08.18 по 07.08.1...,4,4,4
3,"Здравствуйте, интересует жилье с 17.08 по 25.0...",3,3,3
4,Добрый день. Ищем комфортное жильё в Лазаревс...,3,3,3
5,Ищем жильё в Лазаревское заезд 13 июля на 10 д...,4,4,4
6,Здравствуйте! Интересует жилье недорого 1-10 с...,4,4,4
7,Здравствуйте🤗\nИщем жильё в центре рядом с мор...,2,2,2
8,Здравствуйте! Ищем жильё с 15 по 23 августа 2...,3,3,3
9,Сниму жилье в июне у моря 2 взрослых до 1500 в...,2,2,2


### Заявки, на которых улучшенный промпт ошибся

In [31]:
pred_fs = pd.to_numeric(df["result_fewshot"], errors="coerce")
display(df[pred_fs != df["amount"]][["text", "amount", "result_fewshot"]])

,text,amount,result_fewshot
